In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "25"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["thtennant/taaf-kaggle-source-share-fork",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:

# ---- NUESTRO BANCO contra el 27B servido por vLLM en localhost:1234 ----
# PETICIONES CONCURRENTES. El primer intento las hizo una a una y el kernel murio
# por tiempo: con un 27B y una sola peticion en vuelo, vLLM iba a ~1 tok/s (su log).
# El servidor agrupa por lotes solo si le llegan varias a la vez, que es justo como
# trabaja el harness real (28 juegos concurrentes).
import json, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

RAW = "https://raw.githubusercontent.com/jvilladuque90/arc-prize-2026-arc-agi-3/main"
MODELO = "vrfai/Qwen3.6-27B-FP8"
BASE = "http://127.0.0.1:1234/v1"
CONCURRENCIA = 16
PRESUPUESTO_S = 45 * 60          # tope duro: reporta lo que haya

def bajar(p):
    with urllib.request.urlopen(f"{RAW}/{p}?cb={int(time.time())}", timeout=60) as r:
        return r.read().decode("utf-8")

ITEMS = [json.loads(l) for l in bajar("micro_bench.jsonl").splitlines() if l.strip()]
_mp = {}
exec(compile(bajar("scripts/micro_prompts.py"), "micro_prompts.py", "exec"), _mp)
prompt_plan_words, prompt_effect = _mp["prompt_plan_words"], _mp["prompt_effect"]
normalize, trivial_baselines = _mp["normalize"], _mp["trivial_baselines"]

PLAN = [i for i in ITEMS if i["type"] == "plan_action"]
EFECTO = [i for i in ITEMS if i["type"] == "effect_of_action"]
print(f"banco: {len(PLAN)} planificacion + {len(EFECTO)} efecto", flush=True)

dl = time.monotonic() + 900
while time.monotonic() < dl:
    try:
        with urllib.request.urlopen(f"{BASE}/models", timeout=5) as r:
            if r.status == 200:
                break
    except Exception:
        pass
    time.sleep(10)
print("vLLM listo", flush=True)

def preguntar(args):
    prompt, pensar, maxtok = args
    cuerpo = json.dumps({
        "model": MODELO,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0, "max_tokens": maxtok,
        "chat_template_kwargs": {"enable_thinking": bool(pensar)},
    }).encode()
    req = urllib.request.Request(f"{BASE}/chat/completions", data=cuerpo,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=600) as r:
            d = json.loads(r.read())
        return (d["choices"][0]["message"].get("content") or "",
                d.get("usage", {}).get("completion_tokens", 0))
    except Exception as exc:
        return (f"<error {type(exc).__name__}>", 0)

# SALVAGUARDAS (dos corridas perdidas sin saber por que):
#  a) el kernel lleva competition_sources y Kaggle puede abortar si no hay
#     submission.parquet -> se escribe uno vacio ANTES de nada
#  b) resultados a disco tras CADA variante, para que un corte no lo pierda todo
#  c) traceback visible: la corrida anterior murio 80 s despues de que vLLM
#     estuviera listo y el log del kernel no llego a descargarse
import traceback
from pathlib import Path as _P
SALIDA = _P("/kaggle/working/bench27b.json")
try:
    import pandas as _pd
    _pd.DataFrame({"id": [], "output": []}).to_parquet("/kaggle/working/submission.parquet")
    print("submission.parquet vacio escrito (evita que Kaggle aborte)", flush=True)
except Exception as _e:
    print(f"no pude escribir submission.parquet: {_e}", flush=True)

t0 = time.monotonic()
resultado = {"modelo": MODELO, "variantes": {}}

def medir(nombre, items, constructor, tipo, pensar, maxtok):
    if time.monotonic() - t0 > PRESUPUESTO_S:
        print(f"  {nombre}: saltado (presupuesto agotado)", flush=True)
        return
    print(f"  {nombre}: lanzando {len(items)} peticiones ({CONCURRENCIA} en paralelo)...",
          flush=True)
    tareas = [(constructor(it), pensar, maxtok) for it in items]
    with ThreadPoolExecutor(max_workers=CONCURRENCIA) as ex:
        salidas = list(ex.map(preguntar, tareas))
    print(f"  {nombre}: {len(salidas)} respuestas recibidas", flush=True)
    aciertos = sum(1 for it, (txt, _) in zip(items, salidas)
                   if normalize(txt, tipo) == it["answer"])
    toks = [n for _, n in salidas]
    resultado["variantes"][nombre] = {
        "n": len(items), "aciertos": aciertos,
        "precision": round(aciertos / len(items), 3),
        "tokens_medios": round(sum(toks) / max(1, len(toks)), 1),
        "ejemplos": [{"esperado": it["answer"], "crudo": txt.strip()[:120]}
                     for it, (txt, _) in list(zip(items, salidas))[:2]]}
    r = resultado["variantes"][nombre]
    print(f"  {nombre:22} {aciertos:3}/{len(items):3} = {r['precision']:6.1%} | "
          f"{r['tokens_medios']:7.1f} tok | {time.monotonic()-t0:.0f}s", flush=True)
    SALIDA.write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")

# orden por valor: primero lo que compara directo con el 4B (90.9% en planificacion)
for _args in (("B.V3_plan_sin_think", PLAN, prompt_plan_words, "plan_action", False, 64),
              ("B.V3_plan_con_think", PLAN, prompt_plan_words, "plan_action", True, 512),
              ("A.V0_efecto_sin_think", EFECTO, lambda it: prompt_effect(it, False),
               "effect_of_action", False, 64),
              ("A.V0_efecto_con_think", EFECTO, lambda it: prompt_effect(it, False),
               "effect_of_action", True, 512)):
    try:
        medir(*_args)
    except Exception:
        print(f"  {_args[0]}: EXCEPCION", flush=True)
        traceback.print_exc()

print("\n===== BENCH 27B =====")
print(json.dumps(resultado, indent=2, ensure_ascii=False)[:6000])
